<div align="center">

<img src="https://capsule-render.vercel.app/api?type=waving&height=280&color=gradient&text=𝗜𝗺𝗮𝗴𝗲%20Upscaler&fontAlignY=30&fontSize=100&desc=Next-Gen%20FP8%20·%20ComfyUI%20Backend%20·%20Smart%20Caching&descSize=30" />

<br/><br/>

[![Open in Colab](https://img.shields.io/badge/Google_Colab-F9AB00?style=for-the-badge&logo=googlecolab&logoColor=black)](https://colab.research.google.com/github/festverse/Image-Upscaler/blob/main/notebook/ZImagePro.ipynb)
![Model](https://img.shields.io/badge/Model-Z--Image%20Turbo%20Pro-A855F7?style=for-the-badge&logo=huggingface&logoColor=white)
![ComfyUI](https://img.shields.io/badge/Powered%20by-ComfyUI-FF6F00?style=for-the-badge)
![GPU](https://img.shields.io/badge/GPU-T4%20Required-76B900?style=for-the-badge&logo=nvidia&logoColor=white)
![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)
![License](https://img.shields.io/badge/License-MIT-blue?style=for-the-badge&logo=gnu&logoColor=white)

</div>

## 🚀 What is image Turbo Pro?

Next-gen FP8 diffusion pipeline with ComfyUI backend and smart caching. Professional-grade image generation on free Colab.

> **Why?** FP8 quantization cuts VRAM usage nearly in half while preserving output quality — enabling pro-grade generation on free hardware.

**Key Features:**
- 🔋 **GPU Ready** — Runs on free T4 Colab
- ⚡ **FP8 Optimized** — Half the VRAM
- 💾 **Smart Cache** — Models cached in Drive, instant on restart
- 🎯 **One-Click** — Zero configuration

---

### 📐 Supported Resolutions

| Ratio | Resolution | Best For |
|:-----:|:----------:|----------|
| 1:1 | 1024×1024 | Avatars, social posts |
| 16:9 | 1280×720 | Thumbnails, wallpapers |
| 9:16 | 720×1280 | Mobile, stories |
| 4:3 | 1152×864 | Classic photo |
| 21:9 | 1344×576 | Ultrawide, cinematic |

---

### ⚙️ How It Works

```mermaid
flowchart LR
    A[📝 Input] --> B[Process]
    B --> C[Generate]
    C --> D[💾 Output]
```

In [ ]:
#@title 🛠️ 1. Initialize
#@markdown <span style='color:#94a3b8;font-size:13px;'>Set up core environment & download models</span>

from IPython.display import display, HTML
display(HTML('''
<div style="background:#064e3b; color:#6ee7b7; padding:12px 16px; border-radius:8px; font-family:monospace; font-size:14px; margin-bottom:8px;">
  🛠️ <b>Step 1/4</b> — Initialize<br>
  <span style="color:#94a3b8; font-size:12px;">Set up core environment & download models</span>
</div>
'''))

import os, sys, time
_init_start = time.time()

# Clone repo (for Colab) and add src/ to path
REPO_DIR = "/content/ImageUpscaler"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/festverse/Image-Upscaler.git {REPO_DIR} &> /dev/null
    print("   ✓ Repository Cloned")
else:
    !cd {REPO_DIR} && git pull &> /dev/null
    print("   ✓ Repository Updated")

sys.path.insert(0, REPO_DIR)

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

# --- GPU Pre-flight Check ---
import torch
if not torch.cuda.is_available():
    print("   ⚠️ CUDA not detected. Installing CUDA-enabled PyTorch...")
    !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --force-reinstall -q
    import torch  # reload after install
    if not torch.cuda.is_available():
        raise RuntimeError("\n   ✗ Still no CUDA!\n" "\n" "   1. Go to Runtime → Change runtime type → T4 GPU\n" "   2. If already on GPU: Runtime → Disconnect and delete runtime → Re-run all\n")
    print("   ✓ CUDA PyTorch installed")
else:
    print(f"   ✓ CUDA ready: {torch.cuda.get_device_name(0)}")

print("📦 Installing Dependencies (This takes a moment)...")
!cd {LOCAL_WORKSPACE} && pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 -q

from src.config import MODEL_DIRS, UNET_URL, TEXT_ENCODER_URL, VAE_URL
from src.downloader import ensure_aria2, mount_drive, download_file, get_cache_status, check_disk_space

ensure_aria2()

# --- Disk Space Check ---
import shutil
_free = shutil.disk_usage("/content").free / (1024**3)
print(f"   💽 Disk: {_free:.1f} GB free")
if _free < 2:
    print("   ⚠️ Low disk — consider running the 🧹 Cleanup cell below")

# --- Mount Google Drive for persistent cache ---
drive_ok = mount_drive()
model_urls = [UNET_URL, TEXT_ENCODER_URL, VAE_URL]

if drive_ok:
    print("   ✓ Drive mounted")
    status = get_cache_status(model_urls)
    if status["cached"]:
        print(f"   💾 Cache: {len(status['cached'])}/{len(model_urls)} models cached ({status['total_size_gb']:.1f} GB)")
        for m in status["cached"]:
            print(f"      ✓ {m['name'][:45]} ({m['size_gb']:.1f} GB)")
        if status["missing"]:
            for m in status["missing"]:
                print(f"      ↓ {m[:45]} (will download)")
        if status["stale"]:
            print("   ⚠ Cache version outdated — will re-download")
    else:
        print("   📥 No cache yet — first run will download ~7 GB, then cache to Drive")
else:
    print("   ⚠ No Drive — models download fresh each session (slower)")
    print("   💡 Tip: Mount Drive to cache models across sessions")

# --- Download models (cache-first) ---
dirs = {k: os.path.join(LOCAL_WORKSPACE, v) for k, v in MODEL_DIRS.items()}

print("\n📥 Fetching Models...")
download_file(UNET_URL,         dirs["unet"])
download_file(TEXT_ENCODER_URL,  dirs["clip"])
download_file(VAE_URL,           dirs["vae"])

_elapsed = time.time() - _init_start
print(f"\n✅ Environment Ready! (took {_elapsed:.0f}s)")
print("   ➜ Run Cell 2 to load the engine and generate images.")

> ⚠️ **Content Safety Notice**
>
> image is an **unfiltered diffusion model** — it does not have built-in NSFW filters.
> You are **solely responsible** for the content you generate.
>
> **Do NOT** use this tool to create:
> - Illegal content
> - Harmful, abusive, or violent content
> - Non-consensual intimate imagery
> - Content involving minors in any form
>
> By running this notebook, you agree to comply with all applicable laws
> and the [HuggingFace Content Guidelines](https://huggingface.co/content-guidelines).
> The authors assume no liability for misuse.

In [ ]:
#@title 🚀 2. Load Engine & Generate
#@markdown <span style='color:#94a3b8;font-size:13px;'>Load FP8 weights, configure prompt, and generate</span>

from IPython.display import display, HTML
display(HTML('''
<div style="background:#1e3a5f; color:#93c5fd; padding:12px 16px; border-radius:8px; font-family:monospace; font-size:14px; margin-bottom:8px;">
  🚀 <b>Step 2/4</b> — Load Engine & Generate<br>
  <span style="color:#94a3b8; font-size:12px;">Load FP8 weights, configure prompt, and generate</span>
</div>
'''))

import sys, os, time
sys.path.insert(0, "/content/ImageUpscaler")

from src.config import WORKSPACE
from src.generator import load_models, generate_image

os.chdir(WORKSPACE)

# --- Load models (wrapped for clean error display) ---
try:
    nodes, unet_model, clip_model, vae_model = load_models()
except Exception as e:
    display(HTML(f'<div style="background:#7f1d1d; color:#fca5a5; padding:12px 16px; border-radius:8px; font-family:monospace; font-size:13px;">✗ <b>Load Failed:</b><br>{e}</div>'))
    raise SystemExit(1)

# --- Generation Settings ---
positive_prompt = "A cinematic shot of a futuristic neon city, rain reflections, cybernetic aesthetics, 8k masterpiece" # @param {type:"string"}
negative_prompt = "blurry, low quality, text, watermark, distorted" # @param {type:"string"}

#@markdown ### 📐 **Dimensions & Quality**
aspect_ratio = "1280x720 (16:9 Landscape)" # @param ["1024x1024 (1:1 Square)", "1280x720 (16:9 Landscape)", "720x1280 (9:16 Portrait)", "1152x864 (4:3 Photo)", "1344x576 (21:9 Cinema)"]
steps = 20 # @param {type:"slider", min:10, max:50, step:1}
guidance_scale = 1 # @param {type:"slider", min:1.0, max:10.0, step:0.5}

#@markdown ### 🎨 **Sampler & Scheduler**
sampler_name = "euler" # @param ["euler", "euler_ancestral", "res_multistep", "dpmpp_2m", "dpmpp_2m_sde", "dpmpp_3m_sde", "lcm", "ddim", "uni_pc", "heun", "dpm_2", "dpm_2_ancestral", "lms", "dpm_fast", "dpm_adaptive", "dpmpp_2s_ancestral", "dpmpp_sde", "dpmpp_sde_gpu", "dpmpp_2m_sde_gpu", "dpmpp_3m_sde_gpu", "ddpm", "uni_pc_bh2"]
scheduler = "simple" # @param ["simple", "beta", "normal", "karras", "exponential", "sgm_uniform", "ddim_uniform"]

#@markdown ### ⚙️ **Advanced**
seed = -1 # @param {type:"number"}
auto_download = False # @param {type:"boolean"}

# Parse resolution
w, h = [int(x) for x in aspect_ratio.split("(")[0].strip().split("x")]

# --- Generate (wrapped for clean error display) ---
try:
    _gen_start = time.time()
    img, path = generate_image(
        nodes=nodes,
        unet_model=unet_model,
        clip_model=clip_model,
        vae_model=vae_model,
        prompt=positive_prompt,
        negative_prompt=negative_prompt,
        width=w,
        height=h,
        steps=steps,
        cfg=guidance_scale,
        sampler_name=sampler_name,
        scheduler=scheduler,
        seed=seed,
    )
    _gen_elapsed = time.time() - _gen_start

    display(img)
    print(f"   ⏱ Generated in {_gen_elapsed:.1f}s ({w}×{h}, {steps} steps, {sampler_name}/{scheduler})")

except ValueError as e:
    display(HTML(f'<div style="background:#78350f; color:#fbbf24; padding:12px 16px; border-radius:8px; font-family:monospace; font-size:13px;">⚠️ <b>Invalid Setting:</b><br><pre style="white-space:pre-wrap;margin:4px 0 0 0;">{e}</pre></div>'))

except RuntimeError as e:
    display(HTML(f'<div style="background:#7f1d1d; color:#fca5a5; padding:12px 16px; border-radius:8px; font-family:monospace; font-size:13px;">✗ <b>Generation Failed:</b><br><pre style="white-space:pre-wrap;margin:4px 0 0 0;">{e}</pre></div>'))

except Exception as e:
    display(HTML(f'<div style="background:#7f1d1d; color:#fca5a5; padding:12px 16px; border-radius:8px; font-family:monospace; font-size:13px;">✗ <b>Unexpected Error:</b><br><pre style="white-space:pre-wrap;margin:4px 0 0 0;">{e}</pre></div>'))

if auto_download:
    from google.colab import files
    files.download(path)

In [ ]:
#@title 💾 3. Export
#@markdown <span style='color:#94a3b8;font-size:13px;'>Download all results</span>

from IPython.display import display, HTML
display(HTML('''
<div style="background:#3b0764; color:#d8b4fe; padding:12px 16px; border-radius:8px; font-family:monospace; font-size:14px; margin-bottom:8px;">
  💾 <b>Step 3/4</b> — Export<br>
  <span style="color:#94a3b8; font-size:12px;">Download all results</span>
</div>
'''))

from src.exporter import zip_outputs, download_zip, get_output_stats

stats = get_output_stats()
print(f"   📊 {stats['count']} images ({stats['total_mb']:.1f} MB)")

zip_path = zip_outputs()
if zip_path:
    download_zip(zip_path)

In [ ]:
#@title 🧹 4. Cache & Cleanup
#@markdown <span style='color:#94a3b8;font-size:13px;'>Manage Drive cache and free disk space</span>

from IPython.display import display, HTML
display(HTML('''
<div style="background:#4a2c0a; color:#fbbf24; padding:12px 16px; border-radius:8px; font-family:monospace; font-size:14px; margin-bottom:8px;">
  🧹 <b>Maintenance</b> — Cache & Cleanup<br>
  <span style="color:#94a3b8; font-size:12px;">Manage Drive cache and free disk space</span>
</div>
'''))

import os, sys, shutil
sys.path.insert(0, "/content/ImageUpscaler")

from src.config import DRIVE_CACHE_DIR
from src.downloader import get_cache_status, clear_cache, mount_drive
from src.exporter import cleanup_outputs, get_output_stats
from src.config import UNET_URL, TEXT_ENCODER_URL, VAE_URL

#@markdown ---
#@markdown ### 🗑️ **What to clean**
clear_drive_cache = False # @param {type:"boolean"}
clear_old_outputs = False # @param {type:"boolean"}
keep_latest_images = 0 # @param {type:"integer"}

# --- Show current status ---
_free = shutil.disk_usage("/content").free / (1024**3)
print(f"💽 Disk: {_free:.1f} GB free")

_out = get_output_stats()
print(f"🖼️ Outputs: {_out['count']} images ({_out['total_mb']:.1f} MB)")

# Check Drive cache
if mount_drive():
    _urls = [UNET_URL, TEXT_ENCODER_URL, VAE_URL]
    _cs = get_cache_status(_urls)
    print(f"💾 Drive cache: {len(_cs['cached'])}/{len(_urls)} models ({_cs['total_size_gb']:.1f} GB)")

    if clear_drive_cache:
        print("\n🗑️ Clearing Drive cache...")
        clear_cache()
else:
    print("💾 Drive: not mounted")

if clear_old_outputs:
    print("\n🧹 Cleaning old outputs...")
    cleanup_outputs(keep_latest=keep_latest_images)

_free_after = shutil.disk_usage("/content").free / (1024**3)
print(f"\n✅ Done — {_free_after:.1f} GB free")

---

<div align="center">

[![Open in Colab](https://img.shields.io/badge/Google_Colab-F9AB00?style=for-the-badge&logo=googlecolab&logoColor=black)](https://colab.research.google.com/github/festverse/Image-Upscaler/blob/main/notebook/ZImagePro.ipynb)
[![Back to ZImagePro](https://img.shields.io/badge/Back_to_ZImagePro-181717?style=for-the-badge&logo=github&logoColor=white)](../../README.md)
[![Documentation](https://img.shields.io/badge/README-238636?style=for-the-badge&logo=github&logoColor=white)](https://github.com/festverse/Image-Upscaler)
[![Issues](https://img.shields.io/badge/Issues-DA3633?style=for-the-badge&logo=github&logoColor=white)](https://github.com/festverse/Image-Upscaler/issues)
[![Profile](https://img.shields.io/badge/Utsav_Vasava-8957E5?style=for-the-badge&logo=github&logoColor=white)](https://github.com/festverse)

<br/><br/>

> All generated outputs are saved in `/content/results`. Download via the file explorer sidebar.

*Licensed under MIT · Made with ⚡ by [Utsav Vasava](https://github.com/festverse)*

</div>